# Train the Card Model: segmentation, long run (Google Colab, ~6 hours)

Same data you collected (`card_dataset.zip`), trained as a **YOLO26n-seg** model (card outlines, not just boxes)
for `TRAIN_HOURS` (6) with **standard YOLO augmentation**, no custom tricks:
- **Geometry:** rotation up to 45 deg, shift, zoom, shear, slight perspective (no flips: a mirrored card isn't real)
- **Lighting / color:** hue, saturation, brightness; plus blur, motion blur, noise, contrast, shadows, JPEG artifacts
- **Mixing pictures:** mosaic (4 pictures in 1), mixup, and **copy-paste** (real card outlines pasted from other
  pictures, which is where overlapping cards come from)

**Before you start:** *Runtime > Change runtime type > T4 GPU* (or better). Keep the tab open; runs save to Google
Drive every epoch, and if Colab disconnects, rerun the cells above training and use **Resume**.

Steps: upload -> boxes to outlines (check them) -> split -> Drive -> train 6 h -> accuracy -> export `best.onnx`.

In [ ]:
!pip install -q -U ultralytics albumentations

import os, glob, random, shutil, zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as the collector
IMGSZ = 416                     # what the Jetson runs
START_WEIGHTS = 'yolo26n-seg.pt'   # newest YOLO, nano, SEGMENTATION. Use 'yolo11n-seg.pt' if this fails to download
TRAIN_HOURS = 6                 # long run: leave a margin before Colab's session limit
os.chdir('/content')

## 1. Upload our dataset
On the Jetson, zip the folder first (in a Jupyter terminal, inside `HackRPI26/`):
```
zip -r card_dataset.zip card_dataset
```
Download that zip to your laptop, then upload it with the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick card_dataset.zip

shutil.rmtree('/content/card_dataset', ignore_errors=True)
with zipfile.ZipFile('card_dataset.zip') as z:
    z.extractall('/content')
print("Images:", len(glob.glob('/content/card_dataset/images/*.jpg')))

## 2. Boxes -> card outlines (automatic)
Inside each labeled box, everything that doesn't look like the table is the card. That gets fit to 4 corners.
If it can't find a clean card shape, it keeps the box (counted below).

In [ ]:
def read_box_label(path, w, h):
    c, xc, yc, bw, bh = open(path).read().split()[:5]
    xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
    return int(c), (xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2)


def card_polygon(img, box, pad=0.12):
    # The card's outline (4 corners, pixels) inside its box. Card = pixels that don't look like the table.
    # The table color is taken from the edge of a slightly bigger crop around the box.
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    px, py = (x2 - x1) * pad, (y2 - y1) * pad
    X1, Y1 = int(max(0, x1 - px)), int(max(0, y1 - py))
    X2, Y2 = int(min(w, x2 + px)), int(min(h, y2 + py))
    crop = cv2.GaussianBlur(img[Y1:Y2, X1:X2], (5, 5), 0)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    edge = np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    table = np.median(edge, axis=0)
    mask = (np.linalg.norm(lab - table, axis=2) > 30).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    box_area = (x2 - x1) * (y2 - y1)
    if contours:
        hull = cv2.convexHull(max(contours, key=cv2.contourArea))
        quad = None
        for eps in (0.02, 0.03, 0.05, 0.08):
            approx = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True)
            if len(approx) == 4:
                quad = approx.reshape(4, 2).astype(np.float32)
                break
        if quad is None:
            quad = cv2.boxPoints(cv2.minAreaRect(hull)).astype(np.float32)
        quad += [X1, Y1]
        if 0.4 * box_area <= cv2.contourArea(quad) <= 1.3 * box_area:
            return quad, True
    # Fallback: the box itself
    return np.array([(x1, y1), (x2, y1), (x2, y2), (x1, y2)], np.float32), False


def write_seg_label(path, items, w, h):
    # items = [(class, polygon pixels)] -> YOLO segmentation format: class x1 y1 x2 y2 ... (0-1)
    with open(path, 'w') as f:
        for c, poly in items:
            pts = ' '.join(f"{x / w:.6f} {y / h:.6f}" for x, y in np.clip(poly, 0, [w - 1, h - 1]))
            f.write(f"{c} {pts}\n")


SRC = '/content/card_dataset'
ALL = '/content/seg_all'
shutil.rmtree(ALL, ignore_errors=True)
os.makedirs(f'{ALL}/images'); os.makedirs(f'{ALL}/labels')
fell_back = []
for img_path in sorted(glob.glob(f'{SRC}/images/*.jpg')):
    name = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    c, box = read_box_label(f'{SRC}/labels/{name}.txt', w, h)
    poly, ok = card_polygon(img, box)
    if not ok:
        fell_back.append(name)
    shutil.copy(img_path, f'{ALL}/images/')
    write_seg_label(f'{ALL}/labels/{name}.txt', [(c, poly)], w, h)
n = len(glob.glob(f'{ALL}/images/*.jpg'))
print(f"Converted {n} images. Card outline found automatically in {n - len(fell_back)}; used the box for {len(fell_back)}.")

Check some outlines (green). A few `BOX FALLBACK` ones are fine; lots of them means the table color is fooling it.

In [ ]:
def read_seg_label(path, w, h):
    items = []
    for line in open(path).read().splitlines():
        v = line.split()
        items.append((int(v[0]), (np.array(v[1:], float).reshape(-1, 2) * [w, h]).astype(np.float32)))
    return items


def draw_items(img, items, title):
    img = img.copy()
    for c, poly in items:
        cv2.polylines(img, [poly.astype(np.int32)], True, (0, 255, 0), 2)
        x, y = poly.min(0).astype(int)
        cv2.putText(img, CLASSES[c], (x, max(y - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title, fontsize=8); plt.axis('off')


names = [os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{ALL}/images/*.jpg')]
show = random.Random(1).sample(names, min(8, len(names))) + fell_back[:4]
plt.figure(figsize=(18, 9))
for i, name in enumerate(show[:12]):
    img = cv2.imread(f'{ALL}/images/{name}.jpg')
    plt.subplot(3, 4, i + 1)
    draw_items(img, read_seg_label(f'{ALL}/labels/{name}.txt', img.shape[1], img.shape[0]),
               name + ('  (BOX FALLBACK)' if name in fell_back else ''))
plt.tight_layout(); plt.show()

## 3. Split into train / val (80 / 20)

In [ ]:
def split_dataset(src, dst, val_frac=0.2, seed=0):
    names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{src}/images/*.jpg'))
    random.Random(seed).shuffle(names)
    n_val = max(1, int(len(names) * val_frac))
    for split, group in [('val', names[:n_val]), ('train', names[n_val:])]:
        os.makedirs(f'{dst}/{split}/images', exist_ok=True)
        os.makedirs(f'{dst}/{split}/labels', exist_ok=True)
        for n in group:
            shutil.copy(f'{src}/images/{n}.jpg', f'{dst}/{split}/images/')
            shutil.copy(f'{src}/labels/{n}.txt', f'{dst}/{split}/labels/')
    return len(names) - n_val, n_val


shutil.rmtree('/content/seg', ignore_errors=True)
n_train, n_val = split_dataset(ALL, '/content/seg')   # same seed as before: same held-out pictures
with open('/content/seg.yaml', 'w') as f:
    f.write(f"path: /content/seg\ntrain: train/images\nval: val/images\nnc: {len(CLASSES)}\nnames: {CLASSES}\n")
print("train:", n_train, " val:", n_val)

## 4. Save runs to Google Drive
Keep the Colab tab open while it trains.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/blackjack_runs'
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs will be saved to", RUN_DIR)

## 5. Train (`TRAIN_HOURS`, about 6 hours)
Every epoch each training picture gets a fresh random mix of the augmentations listed at the top.
`best.pt` keeps the epoch that scores best on the held-out pictures, so the long run can't make the final model worse.

In [ ]:
import albumentations as A

# Pixel-only changes (the card outlines stay exactly right): camera blur, motion, noise, lighting, shadows, JPEG
PIXEL_AUGS = [
    A.Blur(blur_limit=(3, 5), p=0.10),
    A.MotionBlur(blur_limit=(3, 7), p=0.15),        # arm still moving when the picture is taken
    A.GaussNoise(p=0.15),                           # cheap camera sensor noise
    A.RandomBrightnessContrast(p=0.35),
    A.CLAHE(p=0.10),
    A.RandomShadow(p=0.15),                         # the arm / dealer's hand shading the table
    A.ImageCompression(quality_range=(60, 100), p=0.20),
    A.ToGray(p=0.02),
]

model = YOLO(START_WEIGHTS)
model.train(
    data='/content/seg.yaml', imgsz=IMGSZ,
    time=TRAIN_HOURS, epochs=5000, patience=0,     # time wins: the epoch count and LR schedule get fit to the hours
    batch=32, cache='ram', workers=4, seed=0,
    cos_lr=True, warmup_epochs=5, weight_decay=0.0005,
    save_period=25,                                 # extra checkpoint every 25 epochs (Drive), on top of last.pt / best.pt
    # --- standard YOLO augmentations, applied randomly to every picture, every epoch ---
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,              # color / lighting
    degrees=45,                                     # dealt cards are turned
    translate=0.15, scale=0.5, shear=3, perspective=0.0005,   # position, size, camera angle
    fliplr=0.0, flipud=0.0,                         # a mirrored card isn't a real card
    mosaic=1.0, close_mosaic=30,                    # 4 pictures tiled into one; off for the last 30 epochs
    mixup=0.1,                                      # blend two pictures
    copy_paste=0.5, copy_paste_mode='mixup',        # paste real card outlines from OTHER pictures -> overlapping cards
    augmentations=PIXEL_AUGS,
    project=RUN_DIR, name='seg_long', exist_ok=True)
BEST = f'{RUN_DIR}/seg_long/weights/best.pt'

### Resume (only if Colab disconnected mid-training)
Rerun the cells above (not training), set `RESUME = True`, run this.

In [ ]:
RESUME = False   # set True only if Colab disconnected during training

if RESUME:
    BEST = f'{RUN_DIR}/seg_long/weights/best.pt'
    YOLO(f'{RUN_DIR}/seg_long/weights/last.pt').train(resume=True)   # picks up where it stopped, same settings

## 6. Check accuracy
On the held-out real pictures (never trained on), same test as before so you can compare models.

In [ ]:
model = YOLO(BEST)
model.val(data='/content/seg.yaml', imgsz=IMGSZ)

# Held-out real pictures: the most confident card has the right rank
right, total, misses = 0, 0, []
for img in sorted(glob.glob('/content/seg/val/images/*.jpg')):
    true = CLASSES[int(open(img.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split()[0])]
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    pred = CLASSES[int(r.boxes.cls[r.boxes.conf.argmax()])] if len(r.boxes) else 'none'
    right += pred == true; total += 1
    if pred != true:
        misses.append((os.path.basename(img), true, pred))
print(f"Card reading accuracy (held-out pictures): {right}/{total} = {100 * right / max(total, 1):.1f}%")
for m in misses[:15]:
    print("  ", m)

## See some predictions

In [ ]:
imgs = sorted(glob.glob('/content/seg/val/images/*.jpg'))[:6]
plt.figure(figsize=(18, 9))
for i, img in enumerate(imgs):
    r = model(img, imgsz=IMGSZ, conf=0.5, verbose=False)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')
plt.tight_layout(); plt.show()

## See what it trained on (augmented batches)

In [ ]:
# What the model actually trains on: a batch after all the augmentations (saved by Ultralytics)
for p in sorted(glob.glob(f'{RUN_DIR}/seg_long/train_batch*.jpg'))[:3]:
    plt.figure(figsize=(14, 14)); plt.imshow(plt.imread(p)); plt.title(os.path.basename(p)); plt.axis('off'); plt.show()

## 7. Export for the Jetson
ONNX, opset 12. Downloads `best.pt` and `best.onnx`; put them in `HackRPI26/models/` on the Jetson
(replacing the old ones). `card_reader` and `blackjack_game` read the outlines automatically.

In [ ]:
onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print("Exported:", onnx_path)
shutil.copy(onnx_path, RUN_DIR)   # backup copy in Drive
files.download(BEST)
files.download(onnx_path)